# Train My AI Coding Helper

<a href="https://colab.research.google.com/github/tolani007/sft-coding-agent/blob/main/notebooks/sft_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I use this notebook to teach an AI helper to code. I teach it with 6,625 examples of real coding work. Some examples are from my own work. I use a model named Google Gemma 2 9B. I use a tool named Unsloth to make the training fast. I do this on an A100 GPU.

### Things I need to do first:
1. Go to **Runtime > Change runtime type** and pick **A100 GPU**.
2. Add my Hugging Face Token to Colab Secrets. I click the Secrets icon on the left side. I add a secret named `HF_TOKEN`. I turn on Notebook access.

## 1. Install Tools

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install --upgrade datasets

## 2. Log in to Hugging Face

In [ ]:
from google.colab import userdata
import os

# Load token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

from huggingface_hub import login
login(token=hf_token)

## 3. Load the Model
I load the model to use the large memory and high speed of the A100 GPU.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192 # Use more memory on A100
dtype = None # Auto detects fp16/bf16
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-2-9b-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

## 4. Load the Data
I load the data I made. It has examples of how to write code.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("focustiki/sft-coding-agent-traces")
train_data = dataset["train"]
eval_data = dataset["test"]

print(f"Loaded {len(train_data)} training examples and {len(eval_data)} eval examples.")

## 5. Format the Data
I change the data to a format the model can read.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        # The tokenizer automatically applies the chat format and <bos>/<eos> tokens
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts, }

train_dataset = train_data.map(formatting_prompts_func, batched = True)
eval_dataset = eval_data.map(formatting_prompts_func, batched = True)

## 6. Train the Model

In [ ]:
from trl import SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported




trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Must be False to use completion-only masking
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        warmup_steps = 50,
        # max_steps = 60, # Uncomment to do a quick 60-step test run!
        num_train_epochs = 1, # Set to 3 for a full training run
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train()

## 7. Save and Push
When the training is done, I push the new model parts to my Hugging Face account. Then I can use them.

In [ ]:
model.push_to_hub("focustiki/eigentiki", token = hf_token)
tokenizer.push_to_hub("focustiki/eigentiki", token = hf_token)

print("I pushed the files to Hugging Face.")

## 8. Test Your New Agent
Now we test eigentiki. We ask it a question to see if it gives a good answer.

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "system", "content": "You are eigentiki, an expert coding assistant with access to bash and python tools."},
    {"role": "user", "content": "Write a python script to calculate the first 10 Fibonacci numbers and then run it."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("\nEigentiki says:\n")
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 500, use_cache = True)
